### Data Loading and Preprocessing

In this cell, we load and preprocess our image data using Keras' `ImageDataGenerator`. The generators read images directly from the specified directories for training and validation data. Each image is resized to 128x128 pixels and rescaled so pixel values are in the range [0, 1], which helps the model train more efficiently.

- `train_gen` loads images from the training folder, applies preprocessing, and generates batches of data for training.
- `val_gen` does the same for the validation set.

This setup allows us to efficiently feed batches of preprocessed images to our models during training and evaluation.


In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    'final_processed_data/final_processed_data/train',        # <-- use your actual path
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical'
)
val_gen = val_datagen.flow_from_directory(
    'final_processed_data/final_processed_data/val',
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical'
)


Found 36181 images belonging to 5 classes.
Found 4452 images belonging to 5 classes.


### Hyperparameter Tuning for CNN

This cell tests different combinations of CNN hyperparameters, including filter size, kernel size, dropout rate, and learning rate.  
For each combination, a new model is built and trained for 1 epoch.  
Validation accuracy is printed for each run.  
The best-performing hyperparameters are recorded and displayed at the end.

Best Validation Accuracy: 85.33% 

Best params: {'filters': 32, 'kernel_size': 3, 'dropout': 0.3, 'learning_rate': 0.001}


In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# 1) define the hyper‐parameter lists you want to try
filter_list  = [32, 64]
kernel_list  = [3, 5]
dropout_list = [0.3, 0.5]
lr_list      = [1e-3, 5e-4]

best_val_acc = 0
best_params  = {}

# 2) loop over every combination
for f in filter_list:
    for k in kernel_list:
        for d in dropout_list:
            for lr in lr_list:
                print(f"Trying filters={f}, kernel={k}, dropout={d}, lr={lr}")

                # build a fresh model each time
                model = Sequential([
                    Conv2D(f, (k, k), activation='relu', input_shape=(128,128,3)),
                    MaxPooling2D((2,2)),
                    Conv2D(f*2, (k, k), activation='relu'),
                    MaxPooling2D((2,2)),
                    Conv2D(f*4, (k, k), activation='relu'),
                    MaxPooling2D((2,2)),
                    Flatten(),
                    Dense(128, activation='relu'),
                    Dropout(d),
                    Dense(train_gen.num_classes, activation='softmax')
                ])
                model.compile(
                    optimizer=Adam(lr),
                    loss='categorical_crossentropy',
                    metrics=['accuracy']
                )

                # train for a few epochs and capture the best val‐accuracy
                history = model.fit(
                    train_gen,
                    validation_data=val_gen,
                    epochs=1,       # keep it small so the search is quick
                    verbose=1
                )
                val_acc = max(history.history['val_accuracy'])
                print(f" → val_accuracy: {val_acc:.4f}\n")

                # record if this is the best so far
                if val_acc > best_val_acc:
                    best_val_acc = val_acc
                    best_params = {
                        'filters': f,
                        'kernel_size': k,
                        'dropout': d,
                        'learning_rate': lr
                    }

# 3) summary
print("Best val_accuracy:", best_val_acc)
print("Best params:", best_params)


Trying filters=32, kernel=3, dropout=0.3, lr=0.001
1131/1131 [==============================] - 547s 482ms/step - loss: 0.6479 - accuracy: 0.7739 - val_loss: 0.4284 - val_accuracy: 0.8533
 → val_accuracy: 0.8533

Trying filters=32, kernel=3, dropout=0.3, lr=0.0005
1131/1131 [==============================] - 509s 449ms/step - loss: 0.6536 - accuracy: 0.7702 - val_loss: 0.4624 - val_accuracy: 0.8372
 → val_accuracy: 0.8372

Trying filters=32, kernel=3, dropout=0.5, lr=0.001
1131/1131 [==============================] - 504s 444ms/step - loss: 0.6672 - accuracy: 0.7685 - val_loss: 0.4700 - val_accuracy: 0.8446
 → val_accuracy: 0.8446

Trying filters=32, kernel=3, dropout=0.5, lr=0.0005
1131/1131 [==============================] - 510s 449ms/step - loss: 0.6850 - accuracy: 0.7637 - val_loss: 0.4626 - val_accuracy: 0.8423
 → val_accuracy: 0.8423

Trying filters=32, kernel=5, dropout=0.3, lr=0.001
1131/1131 [==============================] - 919s 811ms/step - loss: 0.7092 - accuracy: 0.7494 

### Flattening Images for DNN/KNN

This cell loads all images from the training and validation folders, resizes them to 128x128 pixels, and normalizes the pixel values.  
Each image is then flattened into a 1D array, making the data suitable for DNN and KNN.


In [1]:
import numpy as np
import os
from tensorflow.keras.preprocessing.image import load_img, img_to_array

def load_images_from_folder(folder, target_size=(128,128)):
    images = []
    labels = []
    class_names = sorted(os.listdir(folder))
    class_to_index = {name: idx for idx, name in enumerate(class_names)}
    for class_name in class_names:
        class_folder = os.path.join(folder, class_name)
        for fname in os.listdir(class_folder):
            img_path = os.path.join(class_folder, fname)
            img = load_img(img_path, target_size=target_size)
            img_arr = img_to_array(img) / 255.0  # Normalize
            images.append(img_arr.flatten())
            labels.append(class_to_index[class_name])
    return np.array(images), np.array(labels)

X_train, y_train = load_images_from_folder('final_processed_data/final_processed_data/train')
X_val, y_val = load_images_from_folder('final_processed_data/final_processed_data/val')


### Hyperparameter Tuning for DNN

This cell tunes key hyperparameters for the Deep Neural Network (DNN), including the number of units, dropout rate, and learning rate.  
For each combination, a new DNN is trained for 1 epoch on the flattened image data.  
Validation accuracy for each setting is printed to help identify the best configuration.

Best Validation Accuracy: 66.15%

DNN Parameters: units=256, dropout=0.3, lr=0.0005


In [20]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

units_list = [128, 256]
dropout_list = [0.3, 0.5]
lr_list = [1e-3, 5e-4]

for units in units_list:
    for dropout in dropout_list:
        for lr in lr_list:
            print(f"Testing DNN: units={units}, dropout={dropout}, lr={lr}")
            model = Sequential([
                Dense(units, activation='relu', input_shape=(X_train.shape[1],)),
                Dropout(dropout),
                Dense(units//2, activation='relu'),
                Dropout(dropout),
                Dense(len(np.unique(y_train)), activation='softmax')
            ])
            model.compile(optimizer=Adam(lr),
                          loss='sparse_categorical_crossentropy',
                          metrics=['accuracy'])
            history = model.fit(
                X_train, y_train,
                validation_data=(X_val, y_val),
                epochs=1,
                verbose=1
            )
            val_acc = max(history.history['val_accuracy'])
            print(f"→ val_accuracy: {val_acc:.4f}\n")


Testing DNN: units=128, dropout=0.3, lr=0.001
1131/1131 [==============================] - 67s 59ms/step - loss: 1.3879 - accuracy: 0.5285 - val_loss: 1.0925 - val_accuracy: 0.5382
→ val_accuracy: 0.5382

Testing DNN: units=128, dropout=0.3, lr=0.0005
1131/1131 [==============================] - 65s 57ms/step - loss: 1.1959 - accuracy: 0.5406 - val_loss: 0.9728 - val_accuracy: 0.6521
→ val_accuracy: 0.6521

Testing DNN: units=128, dropout=0.5, lr=0.001
1131/1131 [==============================] - 65s 57ms/step - loss: 1.4021 - accuracy: 0.5261 - val_loss: 1.1471 - val_accuracy: 0.5382
→ val_accuracy: 0.5382

Testing DNN: units=128, dropout=0.5, lr=0.0005
1131/1131 [==============================] - 65s 57ms/step - loss: 1.2914 - accuracy: 0.5242 - val_loss: 1.1458 - val_accuracy: 0.5382
→ val_accuracy: 0.5382

Testing DNN: units=256, dropout=0.3, lr=0.001
1131/1131 [==============================] - 119s 104ms/step - loss: 1.4042 - accuracy: 0.5347 - val_loss: 0.9722 - val_accuracy: 0.

### Preprocessing and Hyperparameter Tuning for KNN

This cell first standardizes and applies PCA to the flattened image data to reduce dimensionality.  
KNN models are then trained and evaluated with different values of K and distance metrics.  
Validation accuracy, confusion matrix, and classification report are printed for each configuration to help identify the best KNN settings.

Best Val Accuracy: 73.25%

Best Parameters: {'k': 15, 'metric': 'manhattan'}

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.decomposition import PCA

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

max_components = min(30, X_train_scaled.shape[0])  # Use 30 or fewer components, or fewer if sample count is low
pca = PCA(n_components=max_components, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)
print("PCA reduced to", X_train_pca.shape[1], "features.")

# Step 3: Tune KNN with classification report
k_list = [3, 5, 7, 11, 15]
metric_list = ['euclidean', 'manhattan']
best_acc = 0
best_params = {}

for k in k_list:
    for metric in metric_list:
        print(f"Testing KNN: K={k}, metric={metric}")
        knn = KNeighborsClassifier(n_neighbors=k, metric=metric)
        knn.fit(X_train_pca, y_train)
        y_pred = knn.predict(X_val_pca)
        acc = accuracy_score(y_val, y_pred)
        print(f"Val accuracy: {acc:.4f}")
        print("Confusion Matrix:\n", confusion_matrix(y_val, y_pred))
        print("Classification Report:\n", classification_report(y_val, y_pred))
        print()

        if acc > best_acc:
            best_acc = acc
            best_params = {'k': k, 'metric': metric}

print(f"Best Val Accuracy: {best_acc:.4f}")
print("Best params:", best_params)

PCA reduced to 30 features.
Testing KNN: K=3, metric=euclidean
Val accuracy: 0.6842
Confusion Matrix:
 [[  98  144   84    0   12]
 [ 140 1978  244    4   30]
 [ 111  275  833    1   65]
 [   5   19    8    3    0]
 [  33  103  127    1  134]]
Classification Report:
               precision    recall  f1-score   support

           0       0.25      0.29      0.27       338
           1       0.79      0.83      0.80      2396
           2       0.64      0.65      0.65      1285
           3       0.33      0.09      0.14        35
           4       0.56      0.34      0.42       398

    accuracy                           0.68      4452
   macro avg       0.51      0.44      0.46      4452
weighted avg       0.68      0.68      0.68      4452


Testing KNN: K=3, metric=manhattan
Val accuracy: 0.6792
Confusion Matrix:
 [[  89  142   91    2   14]
 [ 151 1963  257    4   21]
 [ 110  270  843    0   62]
 [   8   18    6    3    0]
 [  38  107  127    0  126]]
Classification Report:
   

C:\Users\ADVAIT\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\ADVAIT\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\ADVAIT\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Val accuracy: 0.7325
Confusion Matrix:
 [[  44  176  112    0    6]
 [  17 2117  247    0   15]
 [   8  269  985    0   23]
 [   0   25    9    1    0]
 [   0  106  178    0  114]]
Classification Report:
               precision    recall  f1-score   support

           0       0.64      0.13      0.22       338
           1       0.79      0.88      0.83      2396
           2       0.64      0.77      0.70      1285
           3       1.00      0.03      0.06        35
           4       0.72      0.29      0.41       398

    accuracy                           0.73      4452
   macro avg       0.76      0.42      0.44      4452
weighted avg       0.73      0.73      0.70      4452


Best Val Accuracy: 0.7325
Best params: {'k': 15, 'metric': 'manhattan'}
